# Phase 2.3 — Autoencoder Stability Check

Phase 2.2 retrained the autoencoder with the same architecture as
Phase 2.1 and saw AUC jump from 0.5643 to 0.6003 -- that single number
set the ensemble weight to w=0.3 (IF). This notebook checks whether
0.6003 was a stable result or a lucky single run, by retraining the
exact Phase 2.2 config 5 times under different seeds, then re-tunes
the ensemble weight and normalization bounds using the honest (mean)
AE estimate.

In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import torch

BACKEND_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from core.model import Autoencoder, get_cleaned_train_test_data

MODELS_DIR = BACKEND_DIR / "trained_models"

X_train_cleaned, y_train_cleaned, X_test, y_test = get_cleaned_train_test_data()
X_train_legit = X_train_cleaned[y_train_cleaned == 0]

scaler = joblib.load(MODELS_DIR / "scaler.pkl")
X_train_legit_scaled = scaler.transform(X_train_legit)
X_test_scaled = scaler.transform(X_test)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"backend dir on path: {BACKEND_DIR}")
print(f"X_train_legit_scaled shape: {X_train_legit_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")
print(f"test cheater rate: {y_test.mean():.4f}")
print(f"Using device: {device}")

backend dir on path: D:\ARGUS\backend
X_train_legit_scaled shape: (7610, 11)
X_test_scaled shape: (2400, 11)
test cheater rate: 0.1667
Using device: cuda


## Retrain the Phase 2.2 best AE config 5x under different seeds\n\nFixed: bottleneck_dim=5, hidden_dim=8, lr=1e-3, batch_size=128, max_epochs=200, patience=15, same 90/10 legit-only split logic. Only `torch.manual_seed` varies across runs.

In [2]:
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split as _tts
from torch.utils.data import DataLoader, TensorDataset

BOTTLENECK_DIM = 5
HIDDEN_DIM = 8
LEARNING_RATE = 1e-3
MAX_EPOCHS = 200
PATIENCE = 15

SEEDS = [42, 123, 7, 2024, 99]

stability_runs = []

for seed in SEEDS:
    torch.manual_seed(seed)

    X_ae_train, X_ae_val = _tts(X_train_legit_scaled, test_size=0.10, random_state=42)
    train_tensor = torch.tensor(X_ae_train, dtype=torch.float32)
    val_tensor = torch.tensor(X_ae_val, dtype=torch.float32).to(device)
    train_loader = DataLoader(TensorDataset(train_tensor), batch_size=128, shuffle=True)

    model = Autoencoder(input_dim=11, hidden_dim=HIDDEN_DIM, bottleneck_dim=BOTTLENECK_DIM).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0
    epochs_run = MAX_EPOCHS

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for (batch,) in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            reconstruction = model(batch)
            loss = criterion(reconstruction, batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_reconstruction = model(val_tensor)
            epoch_val_loss = criterion(val_reconstruction, val_tensor).item()

        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            epochs_run = epoch
            break

    model.load_state_dict(best_state)

    model.eval()
    test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
    with torch.no_grad():
        test_reconstruction = model(test_tensor)
        per_sample_mse = torch.mean((test_reconstruction - test_tensor) ** 2, dim=1).cpu().numpy()

    score_min, score_max = per_sample_mse.min(), per_sample_mse.max()
    ae_scores_normalized = (per_sample_mse - score_min) / (score_max - score_min)
    auc = roc_auc_score(y_test, ae_scores_normalized)

    stability_runs.append({
        "seed": seed,
        "epochs_run": epochs_run,
        "final_val_loss": best_val_loss,
        "auc": auc,
        "state_dict": best_state,
    })

    print(f"seed={seed}: epochs_run={epochs_run} final_val_loss={best_val_loss:.6f} AUC-ROC={auc:.4f}")

print()
print(f"{'seed':>6} {'epochs_run':>11} {'final_val_loss':>15} {'AUC-ROC':>10}")
print("-" * 46)
for r in stability_runs:
    print(f"{r['seed']:>6} {r['epochs_run']:>11} {r['final_val_loss']:>15.6f} {r['auc']:>10.4f}")

seed=42: epochs_run=200 final_val_loss=0.293522 AUC-ROC=0.6003


seed=123: epochs_run=83 final_val_loss=0.176974 AUC-ROC=0.5717


seed=7: epochs_run=189 final_val_loss=0.159707 AUC-ROC=0.5786


seed=2024: epochs_run=156 final_val_loss=0.176210 AUC-ROC=0.5699


seed=99: epochs_run=200 final_val_loss=0.152833 AUC-ROC=0.5889

  seed  epochs_run  final_val_loss    AUC-ROC
----------------------------------------------
    42         200        0.293522     0.6003
   123          83        0.176974     0.5717
     7         189        0.159707     0.5786
  2024         156        0.176210     0.5699
    99         200        0.152833     0.5889


## Stability statistics across the 5 runs

In [3]:
aucs = np.array([r["auc"] for r in stability_runs])

mean_auc = float(aucs.mean())
std_auc = float(aucs.std(ddof=1))
min_auc = float(aucs.min())
max_auc = float(aucs.max())

print(f"Mean AUC-ROC: {mean_auc:.4f}")
print(f"Std  AUC-ROC: {std_auc:.4f}")
print(f"Min  AUC-ROC: {min_auc:.4f}")
print(f"Max  AUC-ROC: {max_auc:.4f}")

PHASE_2_2_RESULT = 0.6003
deviation = PHASE_2_2_RESULT - mean_auc
z_like = deviation / std_auc if std_auc > 0 else float("inf")

print()
print(f"Phase 2.2 reported AUC-ROC: {PHASE_2_2_RESULT:.4f}")
print(f"Deviation from this run's mean: {deviation:+.4f} ({z_like:.2f} std devs)")

if PHASE_2_2_RESULT >= max_auc or deviation > std_auc:
    print("Phase 2.2's 0.6003 was a HIGH OUTLIER relative to this 5-run distribution, "
          f"not a stable/typical result (mean {mean_auc:.4f}).")
else:
    print(f"Phase 2.2's 0.6003 was near the mean ({mean_auc:.4f}) -- looks stable.")

Mean AUC-ROC: 0.5819
Std  AUC-ROC: 0.0127
Min  AUC-ROC: 0.5699
Max  AUC-ROC: 0.6003

Phase 2.2 reported AUC-ROC: 0.6003
Deviation from this run's mean: +0.0184 (1.45 std devs)
Phase 2.2's 0.6003 was a HIGH OUTLIER relative to this 5-run distribution, not a stable/typical result (mean 0.5819).


## Select the 'typical' run and save its model\n\nUse the mean AUC across runs as the honest performance estimate (Cell 3), but for the production model artifact, save the single run whose AUC is closest to that mean -- reuses a run already done rather than retraining again.

In [4]:
closest_run = min(stability_runs, key=lambda r: abs(r["auc"] - mean_auc))

print(f"Run closest to mean AUC ({mean_auc:.4f}): seed={closest_run['seed']}, "
      f"AUC={closest_run['auc']:.4f} (diff {abs(closest_run['auc'] - mean_auc):.4f})")

torch.save(closest_run["state_dict"], MODELS_DIR / "autoencoder.pt")
print(f"Saved seed={closest_run['seed']} model state to {MODELS_DIR / 'autoencoder.pt'} "
      f"(overwrites Phase 2.2 file). autoencoder_config.json left as-is -- architecture unchanged.")

typical_ae_model = Autoencoder(input_dim=11, hidden_dim=HIDDEN_DIM, bottleneck_dim=BOTTLENECK_DIM).to(device)
typical_ae_model.load_state_dict(closest_run["state_dict"])
typical_ae_model.eval()

Run closest to mean AUC (0.5819): seed=7, AUC=0.5786 (diff 0.0033)
Saved seed=7 model state to D:\ARGUS\backend\trained_models\autoencoder.pt (overwrites Phase 2.2 file). autoencoder_config.json left as-is -- architecture unchanged.


Autoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=11, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=5, bias=True)
    (3): ReLU()
  )
  (decoder): Sequential(
    (0): Linear(in_features=5, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=11, bias=True)
  )
)

## Recompute ensemble weight with percentile-bounded normalization\n\nUses the existing production Isolation Forest (Phase 2.2 best, n_estimators=300/max_features=0.5, stable/tree-based, AUC 0.5941 -- unaffected by this check) and the typical AE run selected above. Switches from min-max to 1st/99th-percentile-bounded normalization (clipped to [0, 1]), which is the fixed-bounds scheme production scoring will use going forward.

In [5]:
iso_forest = joblib.load(MODELS_DIR / "isolation_forest.pkl")
iso_raw_scores = -iso_forest.decision_function(X_test)

test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
with torch.no_grad():
    test_reconstruction = typical_ae_model(test_tensor)
    ae_raw_scores = torch.mean((test_reconstruction - test_tensor) ** 2, dim=1).cpu().numpy()

if_p1, if_p99 = np.percentile(iso_raw_scores, [1, 99])
ae_p1, ae_p99 = np.percentile(ae_raw_scores, [1, 99])


def normalize(raw, p1, p99):
    return np.clip((raw - p1) / (p99 - p1), 0.0, 1.0)


if_scores_normalized = normalize(iso_raw_scores, if_p1, if_p99)
ae_scores_normalized = normalize(ae_raw_scores, ae_p1, ae_p99)

auc_if_standalone = roc_auc_score(y_test, if_scores_normalized)
auc_ae_standalone = roc_auc_score(y_test, ae_scores_normalized)

print(f"IF score bounds:  p1={if_p1:.6f}  p99={if_p99:.6f}")
print(f"AE score bounds:  p1={ae_p1:.6f}  p99={ae_p99:.6f}")
print()
print(f"IF standalone AUC-ROC (percentile-normalized): {auc_if_standalone:.4f}")
print(f"AE standalone AUC-ROC (percentile-normalized): {auc_ae_standalone:.4f}")
print(f"(Reference -- honest mean AE AUC from 5-run stability check: {mean_auc:.4f})")

weight_results = []
for w in [0.3, 0.4, 0.5, 0.6, 0.7]:
    ensemble_score = w * if_scores_normalized + (1 - w) * ae_scores_normalized
    auc = roc_auc_score(y_test, ensemble_score)
    weight_results.append({"weight": w, "auc": auc})

weight_sorted = sorted(weight_results, key=lambda r: r["auc"], reverse=True)

print()
print(f"{'weight (IF)':>12} {'AUC-ROC':>10}")
print("-" * 24)
for r in weight_sorted:
    print(f"{r['weight']:>12} {r['auc']:>10.4f}")

best_weight = weight_sorted[0]
print()
print(f"Best ensemble weight (honest/corrected): w={best_weight['weight']} (IF weight), "
      f"AUC-ROC={best_weight['auc']:.4f}")
print(f"Phase 2.2 chosen weight was w=0.3, AUC-ROC=0.5987 (based on the potentially "
      f"inflated single 0.6003 AE run).")

IF score bounds:  p1=-0.121779  p99=0.082997
AE score bounds:  p1=0.015225  p99=1.883205

IF standalone AUC-ROC (percentile-normalized): 0.5941
AE standalone AUC-ROC (percentile-normalized): 0.5786
(Reference -- honest mean AE AUC from 5-run stability check: 0.5819)

 weight (IF)    AUC-ROC
------------------------
         0.3     0.6033
         0.4     0.6029
         0.5     0.6016
         0.6     0.6001
         0.7     0.5987

Best ensemble weight (honest/corrected): w=0.3 (IF weight), AUC-ROC=0.6033
Phase 2.2 chosen weight was w=0.3, AUC-ROC=0.5987 (based on the potentially inflated single 0.6003 AE run).


## Save final scoring config -- single source of truth for production scoring

In [6]:
import json

scoring_config = {
    "if_score_p1": float(if_p1),
    "if_score_p99": float(if_p99),
    "ae_score_p1": float(ae_p1),
    "ae_score_p99": float(ae_p99),
    "ensemble_weight_if": best_weight["weight"],
}

scoring_config_path = MODELS_DIR / "scoring_config.json"
with open(scoring_config_path, "w", encoding="utf-8") as f:
    json.dump(scoring_config, f, indent=2)

print(f"Saved {scoring_config_path}")
print(json.dumps(scoring_config, indent=2))
print()
print("Phase 3.1 should read this file rather than recomputing normalization "
      "bounds or the ensemble weight.")

Saved D:\ARGUS\backend\trained_models\scoring_config.json
{
  "if_score_p1": -0.12177912083087014,
  "if_score_p99": 0.08299678789178934,
  "ae_score_p1": 0.015224658520892262,
  "ae_score_p99": 1.8832051980495212,
  "ensemble_weight_if": 0.3
}

Phase 3.1 should read this file rather than recomputing normalization bounds or the ensemble weight.
